# 02b — Repairing duplicate leakage (val/test vs KB)

```text
Repairing duplicate leakage (run AFTER 02_preprocess_images).

Problem: a val/test image with an identical (SHA-256) or near-identical (dHash) copy in the KB
makes evaluation trivial (the retriever finds the image itself or its twin, hence the right caption).
Repair: the twin images are removed from the KB and replaced by other eligible train2017 images
(deterministic draw). The evaluation images (val/test) do NOT change: they are carefully stratified.
Then: rerun 01b (downloads the replacements), 02, then 02b again (must report "nothing to repair"),
then 03 -> 08. Embeddings already computed are reused image by image (04).
```

> Project: comparative study of RAG techniques for image captioning (MS COCO) — **group G1**. Run the notebooks in order: 00 → 01 → 01b → 02 → … → 09. The shared helpers are in `common.py`, `visual.py`, `rag_data.py`.

In [10]:
import os
# --- parameter: "config.yaml" = real COCO data; "config_test.yaml" = synthetic mini-COCO (offline test)
CONFIG = os.environ.get("RAG_CONFIG", "config.yaml")
os.environ["RAG_CONFIG"] = CONFIG
print("config used:", CONFIG)

config utilisée : config.yaml


## Imports et constantes

In [11]:
import argparse
import random
from collections import Counter, defaultdict
from common import (complexity_label, context_label, get_logger, load_config, mkdirs, paths, read_json, set_seed,
                    sha256_ids, write_json)
log = get_logger("dedup")
KEYS = ("image_id", "source", "file_name", "coco_url", "width", "height", "n_objects", "complexity", "context", "interaction")

### `eligible_pool()`
Eligible train2017 images — same rules as 01_make_splits.

In [12]:
def eligible_pool(ann_dir, cfg):
    """Eligible train2017 images — same rules as 01_make_splits."""
    caps = read_json(ann_dir / "captions_train2017.json")
    inst = read_json(ann_dir / "instances_train2017.json")
    cat = {c["id"]: c for c in inst["categories"]}
    n_caps = Counter(a["image_id"] for a in caps["annotations"])
    by_obj = defaultdict(list)
    for a in inst["annotations"]:
        if a.get("iscrowd", 0) == 0:
            by_obj[a["image_id"]].append(a)
    s, pool = cfg["sampling"], {}
    for im in inst["images"]:
        iid = im["id"]
        if n_caps[iid] < cfg["captions_per_image"] or len(by_obj[iid]) < s["min_objects"] or min(im["width"], im["height"]) < s["min_side_px"]:
            continue
        cats = sorted({a["category_id"] for a in by_obj[iid]})
        supers = sorted({cat[c]["supercategory"] for c in cats})
        n_obj = len(by_obj[iid])
        interaction = (1 in cats and len(cats) >= 2) or (sum(a["category_id"] == 1 for a in by_obj[iid]) >= 2) \
            or (len(supers) >= 2 and "animal" in supers)
        pool[iid] = {"image_id": iid, "source": "train2017", "file_name": im["file_name"], "coco_url": im.get("coco_url", ""),
                     "width": im["width"], "height": im["height"], "n_objects": n_obj, "complexity": complexity_label(n_obj),
                     "context": context_label(supers), "interaction": bool(interaction), "cats": cats}
    return pool, {c["id"]: c["name"] for c in cat.values()}

### `find_conflicts()`
Returns ({KB_id_to_remove: reason}, [cases to handle by hand]).

In [13]:
def find_conflicts(audit, ids, thr):
    """Returns ({KB_id_to_remove: reason}, [cases to handle by hand])."""
    drop, manual = {}, []
    for g in audit["exact_duplicates"]:
        ev = [i for i, s in zip(g["image_ids"], g["splits"]) if s in ("val", "test")]
        kb = sorted(i for i, s in zip(g["image_ids"], g["splits"]) if s == "kb")
        if ev:
            for i in kb:
                drop[i] = "copie identique (SHA-256) d'une image val/test"
            if len(ev) > 1:
                manual.append({"type": "doublon exact entre images d'évaluation", **g})
        elif len(kb) > 1:
            for i in kb[1:]:
                drop[i] = "copie identique (SHA-256) dans la KB"
    for n in audit["near_duplicates_eval_vs_others"]:
        if n["match_split"] == "kb":
            drop.setdefault(n["match_id"], f"quasi-doublon (dHash <= {thr}) d'une image val/test")
        else:
            manual.append({"type": "quasi-doublon entre images d'évaluation", **n})
    return {i: r for i, r in drop.items() if i in ids["kb"]}, manual

### `summarize()`

In [14]:
def summarize(pool, ids, cat_names):
    c, ctx, inter, cover = Counter(), Counter(), 0, Counter()
    for i in ids:
        c[pool[i]["complexity"]] += 1; ctx[pool[i]["context"]] += 1; inter += pool[i]["interaction"]; cover.update(pool[i]["cats"])
    return {"n": len(ids), "complexity": dict(c), "context": dict(ctx), "interaction_frac": inter / max(1, len(ids)),
            "classes_covered": len(cover), "min_images_per_class": min(cover.values()) if cover else 0,
            "classes_missing": [cat_names[k] for k in cat_names if k not in cover]}

### `main()`

In [15]:
def main():
    argparse.ArgumentParser().parse_args()
    cfg = load_config(); p = paths(cfg); mkdirs(p); set_seed(cfg["seed"])
    audit = read_json(p["audit"] / "image_audit.json")
    kbj = read_json(p["splits"] / "kb.json")
    ids = {s: {r["image_id"] for r in read_json(p["splits"] / f"{s}.json")["images"]} for s in ("kb", "val", "test")}
    drop, manual = find_conflicts(audit, ids, audit["near_dup_threshold_hamming"])
    for m in manual:
        log.warning("TO HANDLE BY HAND (between evaluation images): %s", m)
    if not drop:
        log.info("Nothing to repair: no val/test image has a twin in the KB.")
        return
    for i, why in sorted(drop.items()):
        log.info("KB image %d removed: %s", i, why)

    hist_f = p["audit"] / "dedup_repair.json"
    hist = read_json(hist_f) if hist_f.exists() else {"rounds": [], "all_dropped": []}
    pool, cat_names = eligible_pool(p["raw"] / "annotations", cfg)
    used = ids["kb"] | ids["val"] | ids["test"] | set(hist["all_dropped"]) | set(drop)
    cand = sorted(i for i in pool if i not in used)
    random.Random(cfg["seed"] + 4 + len(hist["rounds"])).shuffle(cand)
    new_ids = cand[:len(drop)]
    assert len(new_ids) == len(drop), "not enough eligible images to replace"

    recs = [r for r in kbj["images"] if r["image_id"] not in drop] + [{k: pool[i][k] for k in KEYS} for i in new_ids]
    recs.sort(key=lambda r: r["image_id"])
    kbj["images"] = recs
    write_json(p["splits"] / "kb.json", kbj)

    meta = read_json(p["splits"] / "splits_meta.json")
    kb_ids = [r["image_id"] for r in recs]
    meta["id_hash_sha256"]["kb"] = sha256_ids(kb_ids)
    meta["summary"]["kb"] = summarize(pool, kb_ids, cat_names)
    meta.setdefault("dedup_repairs", []).append({"dropped_kb_ids": sorted(drop), "added_kb_ids": sorted(new_ids)})
    write_json(p["splits"] / "splits_meta.json", meta)

    hist["rounds"].append({"dropped": {str(k): v for k, v in sorted(drop.items())}, "added": sorted(new_ids)})
    hist["all_dropped"] = sorted(set(hist["all_dropped"]) | set(drop))
    write_json(hist_f, hist)
    log.info("KB repaired: %d image(s) removed, %d added -> %s", len(drop), len(new_ids), sorted(new_ids))
    log.info("NEXT: rerun 01b_download_images, 02_preprocess_images, then 02b (must say 'nothing to repair'), then 03 -> 08.")

## Run the step

In [16]:
import sys
sys.argv = ["notebook"]          # no CLI arguments in a notebook
try:
    main()
except SystemExit as e:               # certains scripts terminent par sys.exit(code)
    print("done, exit code:", e.code)

19:17:19 | INFO | Rien à réparer : aucune image val/test n'a de jumeau dans la KB.


## Results

In [17]:
import json, pandas as pd
from IPython.display import Image, Markdown, display
from common import load_config, paths
cfg = load_config(); P = paths(cfg)

### Bilan

In [18]:
f = P["audit"] / "dedup_repair.json"
print(json.dumps(json.load(open(f)), indent=1, ensure_ascii=False) if f.exists() else "No repair performed.")
a = json.load(open(P["audit"] / "image_audit.json"))
print("dernier audit (02) -> doublons exacts :", len(a["exact_duplicates"]), "| quasi-doublons val/test vs reste :", len(a["near_duplicates_eval_vs_others"]))

{
 "rounds": [
  {
   "dropped": {
    "186275": "copie identique (SHA-256) d'une image val/test",
    "463132": "quasi-doublon (dHash <= 4) d'une image val/test"
   },
   "added": [
    149053,
    474404
   ]
  }
 ],
 "all_dropped": [
  186275,
  463132
 ]
}
dernier audit (02) -> doublons exacts : 0 | quasi-doublons val/test vs reste : 0
